# 03 — Exploratory Data Analysis (EDA)

**Objective:** Visual exploration of trends, distributions, segments, and patterns in the cleaned World Cup Players dataset.

**Input:** `data/processed/cleaned_worldcup_players.csv`  
**Visualizations:** 10 charts covering team performance, player events, coaching patterns, and match structure.

## 3.1 — Setup & Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

DATA_PATH = PROJECT_ROOT / "data/processed/cleaned_worldcup_players.csv"
print(f"Loading cleaned data from: {DATA_PATH}")

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# Set a consistent dark plot style
sns.set_theme(style="darkgrid", palette="Set2", font_scale=1.1)
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

In [ ]:
df.describe(include="all").T

---
## 3.2 — Chart 1: Top 15 Countries by Total Player Appearances

In [ ]:
team_appearances = (
    df["team_initials"]
    .value_counts()
    .head(15)
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 7))
team_appearances.plot(kind="barh", color=sns.color_palette("viridis", 15), edgecolor="black", ax=ax)
ax.set_title("Top 15 Countries by Total Player Appearances in World Cups", fontweight="bold")
ax.set_xlabel("Total Player-Match Appearances")
ax.set_ylabel("Country")
for i, v in enumerate(team_appearances.values):
    ax.text(v + 20, i, f"{v:,}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

**Insight:** Brazil, Germany, Argentina, and Italy dominate in terms of total player-match appearances, reflecting their consistent World Cup qualification and deep tournament runs.

---
## 3.3 — Chart 2: Distribution of Goals Scored per Player-Match

In [ ]:
# Focus on rows with at least 1 goal
goals_data = df[df["goals"] > 0]

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=goals_data, x="goals", palette="Oranges_d", edgecolor="black", ax=ax)
ax.set_title("Distribution of Goals per Player-Match (Only Goalscorers)", fontweight="bold")
ax.set_xlabel("Goals in a Single Match")
ax.set_ylabel("Number of Player-Match Records")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

print(f"Total goalscoring instances: {len(goals_data):,}")
print(f"Total goals: {df['goals'].sum():,}")
print(f"Max goals in a single match by one player: {df['goals'].max()}")

**Insight:** Most goalscorers score exactly 1 goal per match. Hat-tricks and beyond are exceedingly rare in World Cup history.

---
## 3.4 — Chart 3: Starting XI vs Substitutes

In [ ]:
lineup_counts = df["line_up"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ["#2ecc71", "#e74c3c"]
axes[0].pie(lineup_counts, labels=lineup_counts.index, autopct="%1.1f%%",
            colors=colors, startangle=90, textprops={"fontsize": 12},
            wedgeprops={"edgecolor": "black", "linewidth": 1})
axes[0].set_title("Line-up Split (Pie Chart)", fontweight="bold")

# Bar chart
sns.barplot(x=lineup_counts.index, y=lineup_counts.values, palette=colors,
            edgecolor="black", ax=axes[1])
axes[1].set_title("Line-up Split (Bar Chart)", fontweight="bold")
axes[1].set_ylabel("Count")
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.show()

**Insight:** The split is roughly even between Starting and Substitute entries, reflecting that squad lists include both the starting XI and bench players for each match.

---
## 3.5 — Chart 4: Top 15 Countries by Yellow Cards

In [ ]:
cards_by_team = (
    df.groupby("team_initials")[["yellow_cards", "red_cards"]]
    .sum()
    .sort_values("yellow_cards", ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(14, 7))
cards_by_team.plot(kind="bar", color=["#f1c40f", "#e74c3c"],
                   edgecolor="black", ax=ax)
ax.set_title("Top 15 Countries: Yellow & Red Cards in World Cup History", fontweight="bold")
ax.set_xlabel("Country")
ax.set_ylabel("Total Cards")
ax.legend(["Yellow Cards", "Red Cards"], loc="upper right")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Insight:** Teams with higher appearances naturally accumulate more cards, but some teams show disproportionately high card rates, suggesting a more aggressive playing style.

---
## 3.6 — Chart 5: Position Distribution

In [ ]:
position_counts = df["position"].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(position_counts.index, position_counts.values,
              color=sns.color_palette("coolwarm", len(position_counts)),
              edgecolor="black")
ax.set_title("Player Position Distribution", fontweight="bold")
ax.set_ylabel("Count")
ax.set_xlabel("Position")
for p in bars:
    ax.annotate(f"{int(p.get_height()):,}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

**Insight:** The vast majority are outfield players. Goalkeeper and Captain are special roles — GK-Captain is very rare.

---
## 3.7 — Chart 6: Top 10 Coaches by Matches Managed

In [ ]:
# Each match has multiple player rows, so count distinct MatchIDs per coach
coach_matches = (
    df.groupby("coach_name")["matchid"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(12, 6))
coach_matches.sort_values(ascending=True).plot(
    kind="barh", color=sns.color_palette("mako", 10), edgecolor="black", ax=ax
)
ax.set_title("Top 10 Coaches by Matches Managed in World Cups", fontweight="bold")
ax.set_xlabel("Number of Matches")
ax.set_ylabel("Coach")
for i, v in enumerate(coach_matches.sort_values(ascending=True).values):
    ax.text(v + 0.2, i, str(v), va="center", fontsize=10)
plt.tight_layout()
plt.show()

**Insight:** The most prolific World Cup coaches have managed across multiple tournament editions, accumulating 20+ matches.

---
## 3.8 — Chart 7: Coach Nationality Distribution (Top 10)

In [ ]:
# Count distinct coaches per nationality (not rows, but unique coaches)
coach_nat = (
    df.drop_duplicates(subset=["coach_name"])
    ["coach_nationality"]
    .value_counts()
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
coach_nat.plot(kind="bar", color=sns.color_palette("flare", 10),
               edgecolor="black", ax=ax)
ax.set_title("Top 10 Nationalities of World Cup Coaches", fontweight="bold")
ax.set_xlabel("Coach Nationality")
ax.set_ylabel("Number of Unique Coaches")
plt.xticks(rotation=45, ha="right")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

**Insight:** Brazil, Argentina, and Italy produce the most World Cup coaches, aligning with their strong football traditions and representation in the tournament.

---
## 3.9 — Chart 8: Goals vs Yellow Cards by Team (Scatter)

In [ ]:
team_agg = df.groupby("team_initials").agg(
    total_goals=("goals", "sum"),
    total_yellows=("yellow_cards", "sum"),
    appearances=("player_name", "count")
).reset_index()

fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(
    team_agg["total_goals"], team_agg["total_yellows"],
    s=team_agg["appearances"] / 10, alpha=0.7,
    c=team_agg["total_goals"], cmap="YlOrRd", edgecolors="black"
)
ax.set_title("Goals vs Yellow Cards by Country (bubble size = appearances)", fontweight="bold")
ax.set_xlabel("Total Goals")
ax.set_ylabel("Total Yellow Cards")

# Label top teams
for _, row in team_agg.nlargest(8, "total_goals").iterrows():
    ax.annotate(row["team_initials"],
                (row["total_goals"], row["total_yellows"]),
                fontsize=9, fontweight="bold", ha="left",
                xytext=(5, 5), textcoords="offset points")

plt.colorbar(scatter, label="Total Goals", ax=ax)
plt.tight_layout()
plt.show()

**Insight:** There is a positive trend — teams that play more matches (and score more goals) also tend to accumulate more yellow cards. Some teams cluster above the trend, suggesting a more physical style of play.

---
## 3.10 — Chart 9: Players with Most World Cup Appearances

In [ ]:
# Appearances = number of distinct matches a player featured in
player_appearances = (
    df.groupby("player_name")["matchid"]
    .nunique()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 7))
player_appearances.sort_values(ascending=True).plot(
    kind="barh", color=sns.color_palette("crest", 15), edgecolor="black", ax=ax
)
ax.set_title("Top 15 Players by World Cup Match Appearances", fontweight="bold")
ax.set_xlabel("Number of Matches")
ax.set_ylabel("Player")
for i, v in enumerate(player_appearances.sort_values(ascending=True).values):
    ax.text(v + 0.2, i, str(v), va="center", fontsize=10)
plt.tight_layout()
plt.show()

**Insight:** The most-capped World Cup players have appeared in 20+ matches, spanning multiple tournament editions over a decade-long international career.

---
## 3.11 — Chart 10: Substitution Patterns

In [ ]:
sub_summary = pd.DataFrame({
    "Count": [
        df["sub_in"].sum(),
        df["sub_out"].sum(),
        len(df) - df["sub_in"].sum() - df["sub_out"].sum() + df[df["sub_in"] & df["sub_out"]].shape[0]
    ]
}, index=["Subbed In", "Subbed Out", "No Substitution"])

fig, ax = plt.subplots(figsize=(8, 5))
colors_sub = ["#27ae60", "#c0392b", "#95a5a6"]
sub_summary["Count"].plot(kind="bar", color=colors_sub, edgecolor="black", ax=ax)
ax.set_title("Substitution Events Overview", fontweight="bold")
ax.set_ylabel("Number of Player-Match Records")
ax.set_xlabel("")
plt.xticks(rotation=0)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

**Insight:** The number of sub-ins closely matches the number of sub-outs (as expected — every substitution involves one player entering and one leaving). The vast majority of player-match records involve no substitution event.

---
## 3.12 — EDA Summary

| # | Chart | Key Finding |
|---|-------|-------------|
| 1 | Top Countries by Appearances | BRA, GER, ARG, ITA dominate |
| 2 | Goal Distribution | Most scorers net 1 goal per match |
| 3 | Starting vs Substitute | ~50/50 split (full squad listed) |
| 4 | Yellow & Red Cards | More matches → more cards; some teams disproportionately carded |
| 5 | Position Distribution | Outfield > GK > Captain > GK-Captain |
| 6 | Top Coaches | Longest-serving coaches managed 20+ WC matches |
| 7 | Coach Nationalities | BRA, ARG, ITA produce most coaches |
| 8 | Goals vs Cards (scatter) | Positive correlation; some teams are outliers |
| 9 | Player Appearances | Top players appeared in 20+ WC matches |
| 10 | Substitution Patterns | Sub-in ≈ Sub-out; most records have no sub event |

✅ **Proceed to Notebook 04 (Statistical Analysis).**